# Group-wise comparisons

Compare the relative abundance and protein signatures of pancreatic cell types across donors and disease groups. Each section contributes one donor observation.

The notebooks require the November 13 Islet Explorer protein release that excludes EpCAM from typing and rebuilds the pancreas UMAP without that marker. The load step checks the release ID and the marker list saved with the embedding.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/04_group_comparisons.ipynb).

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "typing_release.json").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the no-EpCAM Scanpy release folder containing typing_release.json.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})

# Locate the shared plotting module when running from a notebook, its parent, or the Senior checkout.
import sys
helper_paths = [cwd, cwd/"notebooks", cwd/"phenocycler"/"notebooks", cwd/"analysis"/"phenocycler"/"notebooks",
                data.parent/"notebooks"]
helper_dir = next((p for p in helper_paths if (p/"phenotype_signatures.py").is_file()), None)
if helper_dir is None:
    raise FileNotFoundError("Run from the Senior checkout or its analysis/phenocycler/notebooks directory.")
sys.path.insert(0, str(helper_dir))
from phenotype_signatures import (PAN_STRUCTURE, HORMONES, IMMUNE, T_CELL, DISPLAY_NAMES,
                                  donor_heatmap, show_profiles, conflict_summary, validate_release)
release = validate_release(data)

DONOR = "115"                      # Any donor listed in the manifest
DISEASE = "ND"                     # ND, AAb+, or T1D
LEVEL = "primary"                    # primary, detail, immune_class, t_subtype, macrophage_profile
UMAP_MARKERS = ["GCG", "INS", "SST", "Podoplanin", "B3TUBB", "CD56", "CD31", "Ker8_18", "CD3e"]
PAIR = ("INS","SST")
group_order = ["ND","AAb+","T1D"]
group_ink = {"ND":"#23699b","AAb+":"#ab6c06","T1D":"#ae4643"}
counts = pd.read_parquet(data / "donor_counts.parquet").query("tissue == 'Pancreas'").copy()
evidence = pd.read_parquet(data / "donor_markers.parquet").query("tissue == 'Pancreas'").copy()
counts["donor"] = counts.donor.astype(str)
evidence["donor"] = evidence.donor.astype(str)
manifest = pd.read_csv(data / "pancreas_manifest.csv",dtype={"donor":str})
if DONOR not in set(manifest.donor) or DISEASE not in group_order:
    raise ValueError("Choose a donor and group from pancreas_manifest.csv")
type_order = ["Alpha","Beta","Delta","Endocrine (ambiguous subtype)","Endocrine (unresolved subtype)",
              "Exocrine","Immune","Blood vessel","Lymphatic","Nerve","Unresolved (no lineage evidence)"]
inks = {"Alpha":"#cf6c30", "Beta":"#287cac", "Delta":"#4c8b58", "Immune":"#965084",
        "Exocrine":"#c1b7a6", "Endocrine (ambiguous subtype)":"#bda559",
        "Endocrine (unresolved subtype)":"#99853f", "Blood vessel":"#b34b50",
        "Lymphatic":"#68a69a", "Nerve":"#5b558c", "Unresolved (no lineage evidence)":"#868686"}
for label in type_order:
    inks.setdefault(str(label), "#73777d")

## 1. Cell-type composition by donor

Each bar partitions all admitted cells. Short bars within the distributions below mark the group median.

In [ ]:
primary = counts.loc[counts.level.eq("primary")]
wide = primary.pivot_table(index=["disease","donor"], columns="label", values="cells",
                           aggfunc="sum", fill_value=0, observed=True)
assert (wide.sum(axis=1) == primary.groupby(["disease","donor"], observed=True).eligible_cells.first().reindex(wide.index)).all()
frac = wide.div(wide.sum(axis=1), axis=0)
sort = sorted(frac.index, key=lambda pair: (group_order.index(pair[0]), int(pair[1])))
frac = frac.loc[sort]
fig, ax = plt.subplots(figsize=(12,4.9))
bottom = np.zeros(len(frac))
for label in frac.columns:
    ax.bar(np.arange(len(frac)), 100*frac[label], bottom=bottom, width=.83, color=inks[label], label=DISPLAY_NAMES.get(label,label))
    bottom += 100*frac[label].to_numpy()
ax.set(xticks=range(len(frac)), xticklabels=frac.index.get_level_values("donor"), ylim=(0,100),
       ylabel="Admitted cells (%)", xlabel="Pancreas donor", xlim=(-.7,len(frac)-.3))
ax.tick_params(axis="x", rotation=60)
for disease in group_order:
    loc = np.flatnonzero(frac.index.get_level_values("disease") == disease)
    ax.text(loc.mean(),103,disease,ha="center",color=group_ink[disease],weight="bold",size=9)
ax.legend(loc="upper center",bbox_to_anchor=(.5,-.26),ncol=4,frameon=False,fontsize=8)
fig.subplots_adjust(bottom=.33)
plt.show()
display(pd.DataFrame({"donor cells":wide.xs(DONOR,level="donor").iloc[0],
                      "donor %":100*frac.xs(DONOR,level="donor").iloc[0],
                      f"{DISEASE} mean donor %":100*frac.xs(DISEASE).mean()}).round(3))

In [ ]:
# Disease-group means are computed after each donor's fractions.
cohort = counts[counts.level.eq("primary")]
frac_by_donor = cohort.pivot_table(index=["donor","disease"],columns="label",values="fraction_tissue",
                                   aggfunc="first",observed=True).reindex(columns=type_order).fillna(0)
assert np.allclose(frac_by_donor.sum(axis=1), 1.0)
panel = ["Alpha","Beta","Delta","Immune"]
fig, axes = plt.subplots(1,len(panel),figsize=(12,3.7))
for ax,label in zip(axes,panel):
    for i,g in enumerate(group_order):
        v = (100*frac_by_donor.xs(g,level="disease")[label]).sort_index()
        ax.scatter(i+np.linspace(-.15,.15,len(v)), v.values, s=24, color=group_ink[g],alpha=.85)
        ax.hlines(v.median(), i-.24, i+.24, color="#20282b", linewidth=1.6)
    ax.set(xticks=range(3),xticklabels=group_order,title=label,ylim=(0,None))
axes[0].set_ylabel("Admitted cells (%)")
fig.tight_layout()
plt.show()

## 2. Compare disease-group marker expression

Both the dotplot and heatmap use the full signatures of the selected hierarchy level. Row annotations identify how many assessable cells and contributing donors support the population. Exact per-marker denominators are retained in `group_measurements`. The default estimates the mean donor response. Set `WEIGHTING = "cells"` to pool cells. Each selected donor can be inspected independently in notebook 02.

In [ ]:
WEIGHTING = "donors"
group_measurements = {}
for group in group_order:
    group_measurements[group] = show_profiles(counts, evidence, level=LEVEL, disease=group,
                                            weighting=WEIGHTING)
display(group_measurements[DISEASE].reset_index())

## 3. Between-donor protein differences

Each row shows one donor and its assigned phenotype count. A donor with one or two cells is shown explicitly. A measured zero assignment has a 0-cell row and cannot provide a within-phenotype expression mean. Gray marks unmeasured or noncomparable combinations. Full marker-specific counts and positive fractions follow the plots.

In [ ]:
from phenotype_signatures import marker_panels
PHENOTYPE = "Beta" if LEVEL in ("primary","detail") else {
    "immune_class":"Imm-T-cell", "t_subtype":"T-CD8", "macrophage_profile":"CD68+CD163+Macrophage_LGALS3+CD4+"
}[LEVEL]
available_labels = counts.loc[counts.level.eq(LEVEL),"label"].unique()
if PHENOTYPE not in available_labels:
    raise ValueError(f"Select a phenotype from {sorted(available_labels)}")
for name, labels, markers in marker_panels("Pancreas", LEVEL, [PHENOTYPE]):
    if PHENOTYPE in labels:
        selected = donor_heatmap(counts, evidence, tissue="Pancreas", level=LEVEL,
                                 label=PHENOTYPE, markers=markers)
display(selected.loc[selected.index.get_level_values("marker").isin(markers)])

### Remaining endocrine conflicts by disease group

Compare like hormone pairs and native decision reasons across sections.

In [ ]:
for group in group_order:
    print(group)
    display(conflict_summary(data, counts, disease=group))